In [221]:
import pandas as pd
import sklearn.linear_model as lm
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# STEP 1: Load training data
titanic_data = pd.read_csv ("./Data/train.csv", index_col="PassengerId")

In [57]:
# STEP 2: Data Exploration
titanic_data.shape

(891, 11)

**Observation**
- dataset contains 891 observation and 11 variables

In [181]:
# y = titanic_data.Survived
# X = titanic_data.drop (["Survived"], axis="columns")
# titanic_data.drop (["Survived"], axis="columns", inplace=True)

In [182]:
# names of variables
titanic_data.columns

Index(['Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket',
       'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [183]:
# identify categorical data continous data
## numberical columns
numerical_col = [cname for cname in titanic_data.columns if titanic_data[cname].dtype in ["int64", "float64"]]

## categorical columns
categorical_col_full = [cname for cname in titanic_data.columns if titanic_data[cname].dtype == "object"]

## categorical columns with low cardinality
categorical_col = [cname for cname in categorical_col_full if titanic_data[cname].nunique () < 10]

print ("Total number of numerical columns: ", len (numerical_col))
print ("Total number of categorical columns: ", len (categorical_col_full))
print ("Total number of categorical columns with low cardinality: ", len (categorical_col))
print (categorical_col)
print (categorical_col_full)

# titanic_data.select_dtypes (exclude=["object"])
# help (pd.DataFrame.select_dtypes)

Total number of numerical columns:  6
Total number of categorical columns:  5
Total number of categorical columns with low cardinality:  2
['Sex', 'Embarked']
['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']


In [184]:
# missing data
null_col = titanic_data.isnull ().sum () [titanic_data.isnull ().sum () > 0]

null_col

Age         177
Cabin       687
Embarked      2
dtype: int64

**Observation**:
dataset contains 3 columns with missing data

In [185]:
# summary
titanic_data.describe ()

,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


**Observation**
1. The oldest person onboard is 80yrs old
2. The youngest person onboard is 4months old

In [186]:
# peek
titanic_data.head (3)

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [187]:
data = titanic_data [numerical_col + categorical_col + ["Cabin"]]
data.head (3)

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex,Embarked,Cabin
PassengerId,,,,,,,,,
1,0,3,22.0,1,0,7.2500,male,S,NaN
2,1,1,38.0,1,0,71.2833,female,C,C85
3,1,3,26.0,0,0,7.9250,female,S,NaN


In [212]:
# separated target from predictors
y = data.Survived.values
X = data.drop (["Survived"], axis="columns")
X_train, X_valid, y_train, y_valid = train_test_split (X,y, random_state=0)
numerical_col

['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']

In [217]:
numerical_transformer = SimpleImputer (strategy="most_frequent")
categorical_transformer = Pipeline (steps=[
  ("imputer", SimpleImputer (strategy="constant")),
  ("OneHot", OneHotEncoder (handle_unknown="ignore", sparse=False))
])


preprocessor = ColumnTransformer (
  transformers=[
    ("cat2", LabelEncoder (), ["Cabin"]),
    ("num", numerical_transformer, numerical_col[1:]),
    ("cat", categorical_transformer, categorical_col)
  ]
)

In [208]:

def get_score (C, X_train=X_train, X_valid=X_valid, y_train=y_train, y_valid=y_valid):
  pipeline = Pipeline (
    steps=[
      ("preprocessor", preprocessor),
      ("model", lm.LogisticRegression(C=C, random_state=0, max_iter=300))
    ])
  
  pipeline.fit (X_train, y_train)
  prediction = pipeline.predict (X_valid)
  return mean_absolute_error (y_valid, prediction)


In [164]:
intercepts = [0.01, 0.1, 1.0, 10.0, 100.0, 10000.0]

for C in intercepts:
  print (f"mae of {C}: ", get_score (C))

mae of 0.01:  0.21524663677130046


/home/kunmi/anaconda3/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:762: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


mae of 0.1:  0.20179372197309417


/home/kunmi/anaconda3/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:762: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


mae of 1.0:  0.19730941704035873


/home/kunmi/anaconda3/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:762: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


mae of 10.0:  0.20179372197309417


/home/kunmi/anaconda3/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:762: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


mae of 100.0:  0.20179372197309417
mae of 10000.0:  0.19730941704035873


/home/kunmi/anaconda3/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:762: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [203]:
def get_score_rf (max_leaf_nodes, n_estimators, X_train=X_train, X_valid=X_valid, y_train=y_train, y_valid=y_valid):
  pipeline = Pipeline (
    steps=[
      ("preprocessor", preprocessor),
      ("model", RandomForestRegressor (max_leaf_nodes=max_leaf_nodes, n_estimators=n_estimators, random_state=0))
    ])
  
  pipeline.fit (X_train, y_train)
  prediction = pipeline.predict (X_valid)
  return mean_absolute_error (y_valid, prediction)

In [218]:
# n_estimators = [5, 50, 500, 5000]

# for n in n_estimators:
#   print (f"mae of {n}: ", get_score_rf (n))
get_score_rf (400, 400)

TypeError: fit_transform() takes 2 positional arguments but 3 were given

In [232]:
oE = LabelEncoder  ()
# oE.fit_transform (data.Cabin)
data.Cabin.query ()

PassengerId
1       NaN
2       C85
3       NaN
4      C123
5       NaN
       ... 
887     NaN
888     B42
889     NaN
890    C148
891     NaN
Name: Cabin, Length: 891, dtype: object

In [129]:
# test_data ["Survived"] = titanic_model.predict (test_X)
# # titanic_model.predict (val_X)
# test_data[["PassengerId", "Survived"]].to_csv ("./Solution/titanic_submission.csv", index=False)